In [1]:
!pip install qdrant-client transformers sentencepiece sentence-transformers

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [4]:
# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# 모델 및 토크나이저 초기화
model1 = AutoModelForSeq2SeqLM.from_pretrained("EbanLee/kobart-summary-v3")
tokenizer1 = AutoTokenizer.from_pretrained("EbanLee/kobart-summary-v3")

model2 = AutoModelForSeq2SeqLM.from_pretrained("NLPBada/kobart-chat-persona-extraction-v2")
tokenizer2 = AutoTokenizer.from_pretrained("NLPBada/kobart-chat-persona-extraction-v2")


config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.5k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
def search_disease_info(query_text, limit=3):
    # 주어진 질병 정보에 대한 검색을 수행하는 함수입니다.
    # query_text: 검색할 질병에 대한 텍스트 쿼리입니다.
    # limit: 검색 결과로 반환할 최대 결과 수입니다. 기본값은 3입니다.

    # 입력된 쿼리 텍스트를 벡터로 인코딩합니다.
    # encoder.encode() 메소드를 사용하여 쿼리 텍스트를 벡터 형태로 변환하고,
    # 리스트 형태로 변환하여 query_vector에 저장합니다.
    query_vector = encoder.encode(query_text).tolist()

    # 클라이언트를 통해 'son5'라는 이름의 컬렉션에서 검색을 수행합니다.
    # search() 메소드에 컬렉션 이름, 쿼리 벡터, 그리고 반환할 결과 수의 한계를 전달합니다.
    search_results = client.search(
        collection_name='son5',  # 검색할 컬렉션의 이름
        query_vector=query_vector,  # 인코딩된 쿼리 벡터
        limit=limit  # 반환할 최대 결과 수
    )

    # 검색 결과를 저장할 리스트를 초기화합니다.
    results = []

    # 검색 결과에서 각 항목에 대해 반복합니다.
    for hit in search_results:
        # 각 검색 결과에서 질병, 답변, 유사도 점수를 추출하여 딕셔너리 형태로 저장합니다.
        result = {
            "질병": hit.payload.get("질병", "N/A"),  # '질병' 필드에서 값을 가져오고, 없으면 'N/A'로 설정
            "답변": hit.payload.get("답변", "N/A"),  # '답변' 필드에서 값을 가져오고, 없으면 'N/A'로 설정
            "유사도 점수": hit.score  # 검색 결과의 유사도 점수를 저장
        }

        # 생성한 결과 딕셔너리를 결과 리스트에 추가합니다.
        results.append(result)

    # 최종적으로 결과 리스트를 반환합니다.
    return results


In [7]:
def generate_response_ensemble(query, context):
    # 질의와 컨텍스트를 기반으로 두 개의 모델에서 생성된 응답을 앙상블하여 반환하는 함수입니다.
    # query: 사용자의 질문 텍스트
    # context: 질문과 관련된 추가 정보 또는 배경 텍스트

    # 입력 텍스트 구성
    # 질문과 컨텍스트를 결합하여 모델에 입력할 텍스트를 생성합니다.
    input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"

    # 모델 1: 요약 모델
    # 입력 텍스트를 토큰화하여 텐서 형태로 변환합니다.
    input_ids1 = tokenizer1.encode(input_text, return_tensors="pt", max_length=1024, truncation=True)

    # 모델 1의 출력을 생성합니다. 이 과정에서 기울기 계산을 하지 않도록 설정합니다.
    with torch.no_grad():
        output1 = model1.generate(
            input_ids1,  # 모델에 입력할 토큰 ID
            max_length=256,  # 생성할 최대 토큰 길이
            num_return_sequences=1,  # 반환할 시퀀스 수
            no_repeat_ngram_size=2,  # 반복되는 n-그램을 방지
            top_k=50,  # 상위 k개의 후보 중에서 샘플링
            top_p=0.95,  # 누적 확률이 p 이하인 후보들 중에서 샘플링
            temperature=0.7,  # 샘플링의 온도, 낮을수록 결정적이고 높을수록 다양함
            do_sample=True,  # 샘플링 모드 활성화
        )

    # 생성된 출력 토큰을 디코딩하여 자연어 응답으로 변환합니다.
    response1 = tokenizer1.decode(output1[0], skip_special_tokens=True)

    # 모델 2: 채팅 및 개인 특성 추출 모델
    # 입력 텍스트를 토큰화하여 텐서 형태로 변환합니다.
    input_ids2 = tokenizer2.encode(input_text, return_tensors="pt", max_length=1024, truncation=True)

    # 모델 2의 출력을 생성합니다. 기울기 계산을 하지 않도록 설정합니다.
    with torch.no_grad():
        output2 = model2.generate(
            input_ids2,  # 모델에 입력할 토큰 ID
            max_length=256,  # 생성할 최대 토큰 길이
            num_return_sequences=1,  # 반환할 시퀀스 수
            no_repeat_ngram_size=2,  # 반복되는 n-그램을 방지
            top_k=50,  # 상위 k개의 후보 중에서 샘플링
            top_p=0.95,  # 누적 확률이 p 이하인 후보들 중에서 샘플링
            temperature=0.7,  # 샘플링의 온도
            do_sample=True,  # 샘플링 모드 활성화
        )

    # 생성된 출력 토큰을 디코딩하여 자연어 응답으로 변환합니다.
    response2 = tokenizer2.decode(output2[0], skip_special_tokens=True)

    # 앙상블: 두 모델의 출력을 결합하여 최종 응답을 생성합니다.
    ensemble_response = f"요약 모델 응답:\n{response1}\n\n채팅 모델 응답:\n{response2}"

    # 최종 앙상블 응답을 반환합니다.
    return ensemble_response


In [8]:
def advanced_rag_search():
    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        results = search_disease_info(query)
        if results:
            best_match = results[0]  # 가장 유사도가 높은 결과 선택
            context = best_match['답변']

            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"관련 질병: {best_match['질병']}")
            print(f"유사도 점수: {best_match['유사도 점수']:.4f}")
            print("원본 답변:")
            print(context)
            print("\n앙상블 모델 생성 답변:")
            generated_response = generate_response_ensemble(query, context)
            print(generated_response)
        else:
            print("검색 결과가 없습니다.")
        print("-" * 50)

# 실행
advanced_rag_search()

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

관련 질병: 아킬레스 건염
유사도 점수: 0.8145
원본 답변:
아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.

앙상블 모델 생성 답변:
요약 모델 응답:
컨텍스트: 아킬레스 건염은 주로 발목 부근에서 발생하며 발뒤꿈치의 아드레스건 부분에서 염증이 발생하면 발목이 아프다. 아프지면 종아리로 통증이 전해질 수 있어 걷거나 달리는 동안 통증을 느껴질 수도 있다.

채팅 모델 응답:
나는 아 아킬레스 건염은 주로 발목에서 발생하는 염증성 질환으로 알고 있다,나는 활동 후 발뒤꿈치의 아드 아 물 아 들 아 지 아 간절 아  아 땅 아 질 아 흘러 땅 땅에서 염증이 발생하면 발 목 주변에 강한 통증이 생길 수 있다 아길 아 발 아 염증은 종아리로 전해질 수 있어 발목을 걷거나 달리는 동안 통증 느껴질 때 발목이 걷고 달리는 데 통증을 느껴 질 수 있다. 아킬로레스건염을 의심한다면, 휴식을 취해도 통통 통 통 증상이 지속되면 의사의 진료를 받는 것이 중요합니다. 아찔한 빠

결과 :
 - 요약 모델은 잘 나오는 것으로 보임
 - 채팅 모델이 잘못된 것 같음
 - 모델의 성능 차이로 인해 이루어 진 문제로 보임